In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pyro_cases.run import vae_dict
from pyro_cases.run_set_transformer_favi import train_and_test_set_transformer_favi

In [ ]:
device = torch.device("cuda:4")

In [ ]:
out_dict = train_and_test_set_transformer_favi(task_name="two_moons",
                                                seed=123456,
                                                device=device,
                                                lr=1e-3,
                                                lr_schedule="cosine_annealing",
                                                batch_size=1024,
                                                network_width=64,
                                                steps=1_000,
                                                test_seed=7272,
                                                num_test_obs=1000,
                                                show_progress=True,
                                                return_vae=True,
                                                suppress_error=False)

In [ ]:
plt.plot(out_dict["favi_training_loss"], linewidth=0.1)
plt.xlabel("Steps")
plt.ylabel("FAVI Loss")
plt.yscale("log")
plt.show()

### VSBC

In [ ]:
favi_vsbc = out_dict["favi_vsbc"]

In [ ]:
def plot_vsbc(vsbc_matrix: torch.Tensor):
    theta_dim = vsbc_matrix.shape[0]
    nrows = theta_dim // 4 + 1 if theta_dim % 4 != 0 else theta_dim // 4
    ncols = 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = axes.flatten()
    for i, vsbc_samples in enumerate(vsbc_matrix):
        ax = axes[i]
        sns.kdeplot(vsbc_samples, 
                    ax=ax, 
                    clip=(0, 1),
                    fill=True,
                    bw_adjust=0.7)
        ax.axvline(x=0.5, color="red", linestyle="--")
        ax.set_title(f"theta[{i}]")
    fig.tight_layout()
    fig.show()

In [ ]:
plot_vsbc(favi_vsbc)

### Output

In [ ]:
out_dict.keys()

In [ ]:
out_dict["favi_test_dict_list"].keys()

In [ ]:
out_dict["favi_test_dict_list"]["obs"].shape

In [ ]:
out_dict["favi_test_dict_list"]["true_theta"][:5]

In [ ]:
out_dict["favi_test_dict_list"]["est_mu"][:5]

In [ ]:
out_dict["favi_test_dict_list"]["est_sigma2"][:5]

### Smoke Test

In [ ]:
for i, task_name in enumerate(vae_dict.keys()):
    print(f"[{i}] run task {task_name}")
    out_dict = train_and_test_set_transformer_favi(task_name=task_name,
                                                    seed=123456,
                                                    device=device,
                                                    lr=1e-3,
                                                    lr_schedule="cosine_annealing",
                                                    batch_size=1024,
                                                    network_width=32,
                                                    steps=10,
                                                    test_seed=7272,
                                                    num_test_obs=1000,
                                                    show_progress=False,
                                                    silent=False,
                                                    return_vae=True,
                                                    suppress_error=False)
    print(f"[{i}] task {task_name} passes")
    print()